In [ ]:
import pandas as pd
import pickle
import os
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.feature_selection import SelectFromModel
from sklearn import metrics
from sklearn import preprocessing
from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_recall_curve
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from inspect import signature
from matplotlib.ticker import FormatStrFormatter
from sklearn.model_selection import train_test_split
import matplotlib.lines as mlines
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn import metrics
import seaborn as sns
from sklearn.metrics import average_precision_score
from matplotlib.lines import Line2D
import scipy.io as sio
from sklearn.impute import KNNImputer
from scipy import stats
from sklearn.utils import resample
import umap
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from BorutaShap import BorutaShap, load_data

In [ ]:
def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True):
    """
    given a sklearn confusion matrix (cm), make a nice plot

    Arguments
    ---------
    cm:           confusion matrix from sklearn.metrics.confusion_matrix

    target_names: given classification classes such as [0, 1, 2]
                  the class names, for example: ['high', 'medium', 'low']

    title:        the text to display at the top of the matrix

    cmap:         the gradient of the values displayed from matplotlib.pyplot.cm
                  see http://matplotlib.org/examples/color/colormaps_reference.html
                  plt.get_cmap('jet') or plt.cm.Blues

    normalize:    If False, plot the raw numbers
                  If True, plot the proportions

    Usage
    -----
    plot_confusion_matrix(cm           = cm,                  # confusion matrix created by
                                                              # sklearn.metrics.confusion_matrix
                          normalize    = True,                # show proportions
                          target_names = y_labels_vals,       # list of names of the classes
                          title        = best_estimator_name) # title of graph

    Citiation
    ---------
    http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html

    """
    import matplotlib.pyplot as plt
    import numpy as np
    import itertools

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
    plt.figure(figsize=(8, 6),dpi=300)
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

In [ ]:
X_All = pd.read_csv('X.csv')
y_All = pd.read_csv('y.csv')

In [ ]:
scaler = preprocessing.StandardScaler().fit(X_All)
X = pd.DataFrame(data=scaler.transform(X_All),columns=X_All.columns)
y=y_All

In [ ]:
imputer = KNNImputer(n_neighbors=10).fit(X)
X = pd.DataFrame(data=imputer.transform(X),columns=X.columns)

In [ ]:
def plot_PCA(X,y):
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    principalComponents = pca.fit_transform(X)
    principalDf = pd.DataFrame(data = principalComponents
                 , columns = ['PC1', 'PC2'])
    finalDf = pd.concat([principalDf, y], axis = 1)
    fig = plt.figure(figsize = (8,8))
    ax = fig.add_subplot(1,1,1) 
    ax.set_xlabel('Principal Component 1', fontsize = 15)
    ax.set_ylabel('Principal Component 2', fontsize = 15)
    ax.set_title('2 component PCA', fontsize = 20)
    targets = [0,1]
    colors = ['b', 'r']
    for target, color in zip(targets,colors):
        indicesToKeep = finalDf['class'] == target
        ax.scatter(finalDf.loc[indicesToKeep, 'PC1']
                   , finalDf.loc[indicesToKeep, 'PC2']
                   , c = color
                   , s = 50)
    ax.legend(targets)
    ax.grid()

# PCA

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
principalComponents = pca.fit_transform(X)
principalDf = pd.DataFrame(data = principalComponents
             , columns = ['PC1', 'PC2'])
finalDf = pd.concat([principalDf, y], axis = 1)
fig = plt.figure(figsize = (8,8))
ax = fig.add_subplot(1,1,1) 
ax.set_xlabel('Principal Component 1', fontsize = 15)
ax.set_ylabel('Principal Component 2', fontsize = 15)
ax.set_title('2 component PCA', fontsize = 20)
targets = [0,1]
colors = ['b', 'r']
for target, color in zip(targets,colors):
    indicesToKeep = finalDf['class'] == target
    ax.scatter(finalDf.loc[indicesToKeep, 'PC1']
               , finalDf.loc[indicesToKeep, 'PC2']
               , c = color
               , s = 50)
ax.legend(targets)
ax.grid()

In [ ]:
filtered_features =[]
for x in list(X.columns):
    if '_min_'  in x:
        continue
    if '_max_'  in x:
        continue
    filtered_features.append(x)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
principalComponents = pca.fit_transform(X[filtered_features])
principalDf = pd.DataFrame(data = principalComponents
             , columns = ['PC1', 'PC2'])
finalDf = pd.concat([principalDf, y], axis = 1)
fig = plt.figure(figsize = (8,8))
ax = fig.add_subplot(1,1,1) 
ax.set_xlabel('Principal Component 1', fontsize = 15)
ax.set_ylabel('Principal Component 2', fontsize = 15)
ax.set_title('2 component PCA', fontsize = 20)
targets = [0,1]
colors = ['b', 'r']
for target, color in zip(targets,colors):
    indicesToKeep = finalDf['class'] == target
    ax.scatter(finalDf.loc[indicesToKeep, 'PC1']
               , finalDf.loc[indicesToKeep, 'PC2']
               , c = color
               , s = 50)
ax.legend(targets)
ax.grid()

# Multicollinearity

In [ ]:
X = X[filtered_features] 

In [ ]:
corr = X.corr()

In [ ]:
number_rejected =0
columns = np.full((corr.shape[0],), True, dtype=bool)
for i in range(corr.shape[0]):
    for j in range(i+1, corr.shape[0]):
        if corr.iloc[i,j] >= 0.96:
            print(X.columns[i], X.columns[j])
            number_rejected +=1
            if columns[j]:
                columns[j] = False
selected_columns = X.columns[columns]


In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
principalComponents = pca.fit_transform(X[selected_columns])
principalDf = pd.DataFrame(data = principalComponents
             , columns = ['PC1', 'PC2'])
finalDf = pd.concat([principalDf, y], axis = 1)
fig = plt.figure(figsize = (8,8))
ax = fig.add_subplot(1,1,1) 
ax.set_xlabel('Principal Component 1', fontsize = 15)
ax.set_ylabel('Principal Component 2', fontsize = 15)
ax.set_title('2 component PCA', fontsize = 20)
targets = [0,1]
colors = ['b', 'r']
for target, color in zip(targets,colors):
    indicesToKeep = finalDf['class'] == target
    ax.scatter(finalDf.loc[indicesToKeep, 'PC1']
               , finalDf.loc[indicesToKeep, 'PC2']
               , c = color
               , s = 50)
ax.legend(targets)
ax.grid()

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
principalComponents = pca.fit_transform(X[selected_columns])
principalDf = pd.DataFrame(data = principalComponents
             , columns = ['PC1', 'PC2'])
finalDf = pd.concat([principalDf, y], axis = 1)
fig = plt.figure(figsize = (8,8))
ax = fig.add_subplot(1,1,1) 
ax.set_xlabel('Principal Component 1', fontsize = 15)
ax.set_ylabel('Principal Component 2', fontsize = 15)
ax.set_title('2 component PCA', fontsize = 20)
targets = [0,1]
colors = ['b', 'r']
for target, color in zip(targets,colors):
    indicesToKeep = finalDf['class'] == target
    ax.scatter(finalDf.loc[indicesToKeep, 'PC1']
               , finalDf.loc[indicesToKeep, 'PC2']
               , c = color
               , s = 50)
ax.legend(targets)
ax.grid()

# Information Gain

In [ ]:
from sklearn.feature_selection import mutual_info_classif
mi = mutual_info_classif(X,y)
mi_df = pd.DataFrame(mi,index =X.columns)
mi_df.columns =['MI']


In [ ]:
mi_df = mi_df.sort_values(by=['MI'],ascending=False)

In [ ]:
mi_df

In [ ]:
for n in [10,50,100,200,300,500,800]:
    
    X = pd.read_csv('X.csv')
    X = X[list(mi_df[:n].index)]
    y = pd.read_csv('y.csv')['class']
    
    # configure the cross-validation procedure
    cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

    # enumerate splits
    cm_total = [[0,0],[0,0]]
    auc_all = list()
    kappa_all = list()
    f1_all = list()
    f= 0
    y_test_all =np.array([])
    y_prob_all =np.array([])
    y_pred_all =np.array([])
    for train_idx, test_idx in cv_outer.split(X,y):
        f+=1
        # split data
        print('Computing Fold ' + str(f))
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        #imputer after scaler
        scaler = preprocessing.StandardScaler().fit(X_train)
        X_train = pd.DataFrame(data=scaler.transform(X_train),columns=X_train.columns)
        imputer = KNNImputer(n_neighbors=10).fit(X_train)
        X_train = pd.DataFrame(data=imputer.transform(X_train),columns=X_train.columns)

        #scaler = preprocessing.StandardScaler().fit(X_test)
        #imputer = KNNImputer(n_neighbors=10).fit(X_test)
        X_test = pd.DataFrame(data=scaler.transform(X_test),columns=X_train.columns)
        X_test = pd.DataFrame(data=imputer.transform(X_test),columns=X_train.columns)

        # configure the cross-validation procedure
        cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

        # define the model
        model = LogisticRegression(random_state=7,penalty='elasticnet',solver='saga',class_weight='balanced')

        params = { 'C': [10**x for x in range(-3,5)], 'l1_ratio':[0.5,0.6,0.7,0.8,0.9]}
        gd_search = GridSearchCV(model, params, scoring='f1_weighted', n_jobs=-1, cv=cv_inner).fit(X_train, y_train)
        best_params = gd_search.best_params_
        best_model = gd_search.best_estimator_

        y_prob = best_model.predict_proba(X_test)[:,1]
        y_pred =  best_model.predict(X_test)
        auc = metrics.roc_auc_score(y_test, y_prob)
        f1 = metrics.f1_score(y_test, y_pred)
        # store the result
        auc_all.append(auc)
        f1_all.append(f1)
        y_test_all = np.concatenate((y_test_all, y_test))
        y_pred_all = np.concatenate((y_pred_all, y_pred))
        y_prob_all = np.concatenate((y_prob_all, y_prob))

        print("Val Auc:",auc, "Best GS Auc:",gd_search.best_score_, "Best Params:",gd_search.best_params_)
        print('Accuracy Score : ' + str(metrics.accuracy_score(y_test, y_pred)))
        print('Precision Score : ' + str(metrics.precision_score(y_test, y_pred)))
        print('Recall Score : ' + str(metrics.recall_score(y_test, y_pred)))
        print('F1 Score : ' + str(f1))
        kappa = metrics.cohen_kappa_score(y_test,y_pred)
        print('Kappa : ' + str(kappa))
        kappa_all.append(kappa)

        cm = metrics.confusion_matrix(y_test,y_pred,labels=[0, 1])
        cm_total = cm + cm_total

        # summarize the estimated performance of the model
        print('Estimated AUC: %.3f (%.3f)' % (np.mean(auc_all), np.std(auc_all)))
        print('Estimated Kappa: %.3f (%.3f)' % (np.mean(kappa_all), np.std(kappa_all)))
        print('Estimated F1: %.3f (%.3f)' % (np.mean(f1_all), np.std(f1_all)))
    
    plot_confusion_matrix(cm=cm_total,
                          target_names=['Nondementia','Dementia'],
                          title='Confusion matrix',
                          normalize=True)
    report= metrics.classification_report(y_test_all,y_pred_all)
    print(report)
    
    fpr, tpr, thresholds = metrics.roc_curve(y_test_all, y_prob_all) 
    roc_auc = metrics.auc(fpr, tpr)
    plt.figure(dpi=300)
    plt.plot(fpr, tpr, color='darkorange', label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc="lower right")
    plt.show()



In [ ]:
for n in [10,50,100,200,300,500,800]:
    
    X = pd.read_csv('X.csv')
    X = X[list(mi_df[:n].index)]
    y = pd.read_csv('y.csv')['class']
    
    # configure the cross-validation procedure
    cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

    # enumerate splits
    cm_total = [[0,0],[0,0]]
    auc_all = list()
    kappa_all = list()
    f1_all = list()
    f= 0
    y_test_all =np.array([])
    y_prob_all =np.array([])
    y_pred_all =np.array([])
    for train_idx, test_idx in cv_outer.split(X,y):
        f+=1
        # split data
        print('Computing Fold ' + str(f))
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        #imputer after scaler
        scaler = preprocessing.StandardScaler().fit(X_train)
        X_train = pd.DataFrame(data=scaler.transform(X_train),columns=X_train.columns)
        imputer = KNNImputer(n_neighbors=10).fit(X_train)
        X_train = pd.DataFrame(data=imputer.transform(X_train),columns=X_train.columns)

        #scaler = preprocessing.StandardScaler().fit(X_test)
        #imputer = KNNImputer(n_neighbors=10).fit(X_test)
        X_test = pd.DataFrame(data=scaler.transform(X_test),columns=X_train.columns)
        X_test = pd.DataFrame(data=imputer.transform(X_test),columns=X_train.columns)

        # configure the cross-validation procedure
        cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

        # define the model
        model = LogisticRegression(random_state=7,penalty='elasticnet',solver='saga',class_weight='balanced')
        feature_selection_model = RandomForestClassifier(random_state=7,class_weight='balanced')
        pipeline = Pipeline(
        [ ("feature_selection", SelectFromModel(feature_selection_model)),
            ("classification",model)])

        params = { 'classification__C': [10**x for x in range(-3,5)], 'classification__l1_ratio':[0.5,0.6,0.7,0.8,0.9]}
        
        gd_search = GridSearchCV(pipeline, params, scoring='f1_weighted', n_jobs=-1, cv=cv_inner).fit(X_train, y_train)
    
        params = { 'C': [10**x for x in range(-3,5)], 'l1_ratio':[0.5,0.6,0.7,0.8,0.9]}
        
        gd_search = GridSearchCV(model, params, scoring='f1_weighted', n_jobs=-1, cv=cv_inner).fit(X_train, y_train)
        best_params = gd_search.best_params_
        best_model = gd_search.best_estimator_

        y_prob = best_model.predict_proba(X_test)[:,1]
        y_pred =  best_model.predict(X_test)
        auc = metrics.roc_auc_score(y_test, y_prob)
        f1 = metrics.f1_score(y_test, y_pred)
        # store the result
        auc_all.append(auc)
        f1_all.append(f1)
        y_test_all = np.concatenate((y_test_all, y_test))
        y_pred_all = np.concatenate((y_pred_all, y_pred))
        y_prob_all = np.concatenate((y_prob_all, y_prob))

        print("Val Auc:",auc, "Best GS Auc:",gd_search.best_score_, "Best Params:",gd_search.best_params_)
        print('Accuracy Score : ' + str(metrics.accuracy_score(y_test, y_pred)))
        print('Precision Score : ' + str(metrics.precision_score(y_test, y_pred)))
        print('Recall Score : ' + str(metrics.recall_score(y_test, y_pred)))
        print('F1 Score : ' + str(f1))
        kappa = metrics.cohen_kappa_score(y_test,y_pred)
        print('Kappa : ' + str(kappa))
        kappa_all.append(kappa)

        cm = metrics.confusion_matrix(y_test,y_pred,labels=[0, 1])
        cm_total = cm + cm_total

        # summarize the estimated performance of the model
        print('Estimated AUC: %.3f (%.3f)' % (np.mean(auc_all), np.std(auc_all)))
        print('Estimated Kappa: %.3f (%.3f)' % (np.mean(kappa_all), np.std(kappa_all)))
        print('Estimated F1: %.3f (%.3f)' % (np.mean(f1_all), np.std(f1_all)))
    
    plot_confusion_matrix(cm=cm_total,
                          target_names=['Nondementia','Dementia'],
                          title='Confusion matrix',
                          normalize=True)
    report= metrics.classification_report(y_test_all,y_pred_all)
    print(report)
    
    fpr, tpr, thresholds = metrics.roc_curve(y_test_all, y_prob_all) 
    roc_auc = metrics.auc(fpr, tpr)
    plt.figure(dpi=300)
    plt.plot(fpr, tpr, color='darkorange', label='ROC curve (area = %0.2f)' % roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend(loc="lower right")
    plt.show()



In [ ]:
SelectFromModel

In [ ]:
SelectKBest